In [0]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
databricks_root = notebook_path.split("/notebooks/")[0]

if not databricks_root.startswith("/Workspace"):
    databricks_root = "/Workspace" + databricks_root

sys.path.append(databricks_root)

dbutils.widgets.text("topic_name", "user_activity")
topic_name = dbutils.widgets.get("topic_name")

from src.config import TOPIC_CONFIGS
from src.bronze_utils import ingest_bronze

creds = {
    "bootstrap_servers": dbutils.secrets.get(scope="redpanda", key="bootstrap-servers"),
    "username": dbutils.secrets.get(scope="redpanda", key="username"),
    "password": dbutils.secrets.get(scope="redpanda", key="password"),
    "schema_registry_url": dbutils.secrets.get(scope="redpanda", key="schema-registry-url"),
}

config = TOPIC_CONFIGS[topic_name]
ingest_bronze(spark, config, creds)